# Проверка ручного запуска DAG за сентябрь 2026

После запуска `etl__acq_final_df__test` с Logical date `2026-09-30T12:00:00+03:00`.

1. Амортизация за сентябрь (Impala, `tmp_shestopalov_acq_04a_amortization`).
2. КЕДР за сентябрь (Impala, `tmp_shestopalov_acq_kedr_chod`).
3. Тестовая таблица дашборда против боевой по месяцам (DRP).
4. Дубли договоров в тестовой таблице (DRP).

Только чтение. Логин keytab и логин/пароль DRP спрашиваются один раз. Для DRP вводите свой логин DRP так же, как в `qc_acq_dag_readiness.ipynb`; если LDAP отказал, не повторяйте больше 2 раз, чтобы не заблокировать учётку.

In [ ]:
from getpass import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
pd.set_option("display.float_format", "{:,.2f}".format)

AMORT_TABLE = "sandbox_ai.tmp_shestopalov_acq_04a_amortization"
KEDR_TABLE = "sandbox_ai.tmp_shestopalov_acq_kedr_chod"
PROD_TABLE = "sbx_da.tmp_shestopalov_acq_fin_version"
TEST_TABLE = "sbx_da.tmp_shestopalov_acq_fin_version_test"
CHECK_MONTH = "2026-09"

if "imp" not in globals():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={"user_name": input("Login (keytab): ").strip()},
    )


def imp_fetch(sql):
    with imp:
        imp.execute("SET MEM_LIMIT=4g")
        return imp.fetch(sql)


def get_drp():
    global drp
    if "drp" not in globals():
        drp = connect(
            to="DRP",
            user_params={"user_name": input("DRP login: ").strip(), "password": getpass("DRP password: ")},
        )
    return drp


def drp_fetch(sql):
    conn = get_drp()
    with conn:
        return conn.fetch(sql)


with imp:
    imp.execute(f"REFRESH {AMORT_TABLE}")
    imp.execute(f"REFRESH {KEDR_TABLE}")
print("Impala: OK")

## 1. Амортизация за сентябрь

Норма: один месяц `2026-09`, около 21–22 тыс. терминалов, почти все `complete`, `missing_price` около 120 (A8900, MoreFun, Verifone), сумма около 2,9–3,0 млн (как в августе).

In [ ]:
amort = imp_fetch(f"""
    SELECT report_month, qc_status,
           count(*) AS terminals,
           count(DISTINCT c_nter) AS distinct_nter,
           round(sum(amortization_for_report_month), 2) AS amort
    FROM {AMORT_TABLE}
    GROUP BY report_month, qc_status
    ORDER BY report_month, terminals DESC
""")
for c in ["terminals", "distinct_nter", "amort"]:
    amort[c] = pd.to_numeric(amort[c], errors="coerce")
display(amort)

months = sorted(amort["report_month"].astype(str).unique())
total_terms = int(amort["terminals"].sum())
total_amort = float(amort["amort"].sum())
complete_pct = 100 * amort.loc[amort["qc_status"] == "complete", "terminals"].sum() / max(total_terms, 1)
dup = total_terms - int(amort["distinct_nter"].sum())

print(f"Месяцы: {months}")
print(f"Терминалов: {total_terms:,} | complete: {complete_pct:.1f}% | сумма: {total_amort:,.2f}")
print("OK: только сентябрь" if months == [CHECK_MONTH] else f"ВНИМАНИЕ: ожидался только {CHECK_MONTH}")
print("OK: сумма в ожидаемом диапазоне" if 2_500_000 <= total_amort <= 3_500_000 else "ВНИМАНИЕ: сумма вне 2,5–3,5 млн, пришлите вывод")
print("OK: без дублей терминалов" if dup == 0 else f"ВНИМАНИЕ: дублей терминалов {dup}")

print("\nМодели без цены:")
display(imp_fetch(f"""
    SELECT model_device, count(*) AS terminals
    FROM {AMORT_TABLE}
    WHERE qc_status = 'missing_price'
    GROUP BY model_device
    ORDER BY terminals DESC
    LIMIT 20
"""))

## 2. КЕДР за сентябрь

Норма: одна строка `2026-09`, ненулевые `inn_cnt` и `kedr_sum`, ИНН не повторяются.

In [ ]:
kedr = imp_fetch(f"""
    SELECT report_month,
           count(*) AS inn_cnt,
           count(DISTINCT inn) AS inn_distinct,
           round(sum(kedr_obshiy_chod), 2) AS kedr_sum,
           sum(CASE WHEN src_cnt > 1 THEN 1 ELSE 0 END) AS inn_in_several_sources
    FROM {KEDR_TABLE}
    GROUP BY report_month
    ORDER BY report_month
""")
for c in ["inn_cnt", "inn_distinct", "kedr_sum", "inn_in_several_sources"]:
    kedr[c] = pd.to_numeric(kedr[c], errors="coerce")
display(kedr)

ok_month = kedr["report_month"].astype(str).tolist() == [CHECK_MONTH]
ok_sum = len(kedr) > 0 and float(kedr["kedr_sum"].fillna(0).sum()) != 0
ok_dup = len(kedr) > 0 and (kedr["inn_cnt"] == kedr["inn_distinct"]).all()
print("OK: только сентябрь" if ok_month else f"ВНИМАНИЕ: ожидался только {CHECK_MONTH}")
print("OK: сумма КЕДР не ноль" if ok_sum else "ВНИМАНИЕ: сумма КЕДР нулевая или пустая")
print("OK: ИНН не повторяются" if ok_dup else "ВНИМАНИЕ: есть повторы ИНН")

## 3. Тестовая таблица против дашборда (DRP)

Норма:
- январь–август: `rows`, `chod`, `fin_result` совпадают в prod и test (`same_*` = True);
- сентябрь: в test есть строки, `amortization_test` около 2,9–3,0 млн, `kedr_test` не ноль;
- октябрь и ноябрь посчитаны ещё по старой логике, их пересчитает плановый запуск в понедельник.

In [ ]:
cmp = drp_fetch(f"""
WITH src AS (
    SELECT 'prod' AS tbl, report_month, chod, fin_result, amortization, kedr_obshiy_chod_contrib, n_agr
    FROM {PROD_TABLE}
    UNION ALL
    SELECT 'test', report_month, chod, fin_result, amortization, kedr_obshiy_chod_contrib, n_agr
    FROM {TEST_TABLE}
),
num AS (
    SELECT
        tbl,
        substring(btrim(report_month) FROM 1 FOR 7) AS report_month,
        n_agr,
        CASE WHEN chod ~ '^-?[0-9]+(\\.[0-9]+)?$' THEN chod::numeric END AS chod,
        CASE WHEN fin_result ~ '^-?[0-9]+(\\.[0-9]+)?$' THEN fin_result::numeric END AS fin_result,
        CASE WHEN amortization ~ '^-?[0-9]+(\\.[0-9]+)?$' THEN amortization::numeric END AS amortization,
        CASE WHEN kedr_obshiy_chod_contrib ~ '^-?[0-9]+(\\.[0-9]+)?$' THEN kedr_obshiy_chod_contrib::numeric END AS kedr
    FROM src
)
SELECT
    report_month,
    sum(CASE WHEN tbl = 'prod' THEN 1 ELSE 0 END) AS rows_prod,
    sum(CASE WHEN tbl = 'test' THEN 1 ELSE 0 END) AS rows_test,
    round(sum(CASE WHEN tbl = 'prod' THEN chod END), 0) AS chod_prod,
    round(sum(CASE WHEN tbl = 'test' THEN chod END), 0) AS chod_test,
    round(sum(CASE WHEN tbl = 'prod' THEN fin_result END), 0) AS fin_result_prod,
    round(sum(CASE WHEN tbl = 'test' THEN fin_result END), 0) AS fin_result_test,
    round(sum(CASE WHEN tbl = 'prod' THEN amortization END), 0) AS amortization_prod,
    round(sum(CASE WHEN tbl = 'test' THEN amortization END), 0) AS amortization_test,
    round(sum(CASE WHEN tbl = 'test' THEN kedr END), 0) AS kedr_test
FROM num
GROUP BY report_month
ORDER BY report_month
""")
num_cols = [c for c in cmp.columns if c != "report_month"]
cmp[num_cols] = cmp[num_cols].apply(pd.to_numeric, errors="coerce")
for m in ["rows", "chod", "fin_result"]:
    cmp[f"same_{m}"] = (cmp[f"{m}_prod"].fillna(0) - cmp[f"{m}_test"].fillna(0)).abs() < 1
display(cmp)

closed = cmp[cmp["report_month"] < CHECK_MONTH]
bad = closed[~(closed["same_rows"] & closed["same_chod"] & closed["same_fin_result"])]
print("OK: январь–август совпадают" if bad.empty else f"ВНИМАНИЕ: расходятся месяцы {bad['report_month'].tolist()}")

sep = cmp[cmp["report_month"] == CHECK_MONTH]
if sep.empty or sep["rows_test"].iloc[0] == 0:
    print("ВНИМАНИЕ: сентября нет в test")
else:
    s = sep.iloc[0]
    print(f"Сентябрь в test: строк {int(s['rows_test']):,}, ЧОД {s['chod_test']:,.0f}, "
          f"амортизация {s['amortization_test']:,.0f}, КЕДР {s['kedr_test']:,.0f}")

## 4. Дубли договоров в тестовой таблице

Норма: пустая таблица.

In [ ]:
dups = drp_fetch(f"""
    SELECT substring(btrim(report_month) FROM 1 FOR 7) AS report_month, n_agr, count(*) AS cnt
    FROM {TEST_TABLE}
    GROUP BY 1, 2
    HAVING count(*) > 1
    ORDER BY 3 DESC
    LIMIT 20
""")
display(dups)
print("OK: дублей договоров нет" if dups is None or len(dups) == 0 else "ВНИМАНИЕ: есть дубли договоров")